# Lesson 000.23 -- The Deficit Angle: Discrete 2D Curvature vs. 4D Non-Abelian Regge Holonomy

AuthorityEffect: None.

> **Claim-typing legend**
> - **[Pinned]**: reproduced from verified checked-in Regge calculus and Lean 4 specifications (`Level21DiscreteRegge.lean`).
> - **[Executed]**: verified by deterministic Python execution in this notebook.
> - **[Declared]**: pedagogical convention or beginner framing.
> - **[Open]**: questions left for future exploration.

## 1. 2D Vertex Deficit and the Discrete Gauss-Bonnet Theorem

[Pinned] On a triangulated 2D surface, curvature concentrates at **0D vertices**.
The deficit angle $\epsilon(v)$ measures the departure of the incident corner angles from a full turn ($1$ complete turn, or $2\pi$ radians):
$$\epsilon(v) = 1 - \sum_{\sigma \ni v} \tau_v(\sigma).$$

For a **closed, orientable piecewise Euclidean 2-manifold without boundary**, summing deficits across all vertices yields the exact discrete **Gauss-Bonnet identity**:
$$\sum_{v \in V} \epsilon(v) = \chi(M)$$
where $\chi(M) = V - E + F$ is the Euler characteristic.
For surfaces with boundary, boundary vertices carry exterior turning angles $\kappa_g(v) = \frac{1}{2} - \sum \tau_v$, and $\sum_{v \in \text{int}} \epsilon(v) + \sum_{v \in \partial} \kappa_g(v) = \chi(M)$.

In [ ]:
from fractions import Fraction as Q

# Tetrahedron topology
V = 4
E = 6
F = 4
chi = V - E + F
assert chi == 2

# Regular equilateral triangles: corner angle = 1/6 turn (60 degrees)
corner_turn = Q(1, 6)
triangles_at_vertex = 3
sum_angle_v = triangles_at_vertex * corner_turn
assert sum_angle_v == Q(1, 2)  # 1/2 turn = 180 degrees

# Deficit angle at each vertex
deficit_v = Q(1, 1) - sum_angle_v
assert deficit_v == Q(1, 2)

# Total deficit across 4 vertices
total_deficit = V * deficit_v
assert total_deficit == Q(2, 1)  # 2 full turns!
assert total_deficit == chi * Q(1, 1)  # Discrete Gauss-Bonnet holds exactly!

print("2D Vertex deficit and discrete Gauss-Bonnet verified on tetrahedron!")

## 2. 2D vs. 4D Curvature: Hinges, Planes, and Ordered Holonomy in Euclidean 4D Regge Geometry

[Pinned] In Euclidean 4D Regge calculus (Stage 21):
1. **Curvature Resides on 2D Triangular Hinges (Bones):** An individual hinge has a scalar deficit angle and a specified normal rotation 2-plane.
2. **Rotations in $SO(4)$ are Non-Abelian:** For loops traversing multiple hinges in different planes, rotation matrices must be composed in path order.
3. **Formal Specification (`Level21DiscreteRegge.lean`):**
   Formal Lean theorem `scalar_sum_does_not_certify_closure` establishes that a scalar sum of angles can vanish along a closed path:
   $$\theta_1 + \theta_2 - \theta_1 - \theta_2 = 0,$$
   yet the ordered loop product of finite rotation matrices:
   $$R_2 R_1 R_2^{-1} R_1^{-1} \ne I_{4\times 4}$$
   is strictly non-trivial!
   
Higher-dimensional discrete curvature requires path-ordered matrix holonomy, not naive scalar sums.


In [ ]:
def mat_mult(A, B):
    n = len(A)
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(n)] for i in range(n)]


def identity(n):
    return [[1 if i == j else 0 for j in range(n)] for i in range(n)]


# Exact rational 4x4 rotation matrices (quarter-turns in xy and yz planes)
R_xy = [
    [0, -1, 0, 0],
    [1, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
]
R_xy_inv = [
    [0, 1, 0, 0],
    [-1, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
]

R_yz = [
    [1, 0, 0, 0],
    [0, 0, -1, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
]
R_yz_inv = [
    [1, 0, 0, 0],
    [0, 0, 1, 0],
    [0, -1, 0, 0],
    [0, 0, 0, 1],
]

# 1. Scalar sum vanishes
scalar_sum = Q(1, 4) + Q(1, 4) - Q(1, 4) - Q(1, 4)
assert scalar_sum == Q(0)

# 2. Ordered matrix holonomy loop: R_yz * R_xy * R_yz_inv * R_xy_inv
loop_holonomy = mat_mult(mat_mult(mat_mult(R_yz, R_xy), R_yz_inv), R_xy_inv)
I4 = identity(4)

# 3. Lean 4 theorem verified: scalar_sum_does_not_certify_closure
assert loop_holonomy != I4

# Check action on basis vector e0 = [1, 0, 0, 0]
e0 = [1, 0, 0, 0]
rot_e0 = [sum(loop_holonomy[i][j] * e0[j] for j in range(4)) for i in range(4)]
assert rot_e0 == [0, -1, 0, 0]  # Transformed to -e1!
assert rot_e0 != e0

print(
    "4D Non-abelian Regge holonomy demonstrated locally via exact rational matrix commutator [Executed]; consistent with Level21DiscreteRegge.lean specification [Declared]."
)

## 3. Diagnostic Probe (2026 Micro-Attention Check)

**Question:** Why can we NOT determine whether transport around a multi-hinge loop in 4D Euclidean discrete geometry is flat by simply verifying that the scalar sum of dihedral angles around a loop equals zero?

- **A:** Because 4D simplices have no angles.
- **B:** Because rotations in $SO(4)$ do not commute in general; there exist rotations in different planes whose ordered matrix holonomy $R_2 R_1 R_2^{-1} R_1^{-1} \ne I$ remains non-trivial even when the scalar sum of angles is zero.
- **C:** Because floating-point numbers round improperly.

*(Correct answer: **B**)*


In [ ]:
def diagnostic_probe(choice: str) -> bool:
    return choice.strip().upper() == "B"


assert diagnostic_probe("B")
assert not diagnostic_probe("A")
assert not diagnostic_probe("C")

# Student Exercise: Regular Octahedron (V=6, E=12, F=8, chi=2)
# 4 equilateral triangles meet at each vertex
V_oct = 6
E_oct = 12
F_oct = 8
chi_oct = V_oct - E_oct + F_oct
assert chi_oct == 2

triangles_at_v_oct = 4
sum_angle_oct = triangles_at_v_oct * corner_turn
assert sum_angle_oct == Q(4, 6) == Q(2, 3)

deficit_oct = Q(1, 1) - sum_angle_oct
assert deficit_oct == Q(1, 3)

total_deficit_oct = V_oct * deficit_oct
assert total_deficit_oct == Q(2, 1) == chi_oct * Q(1, 1)

print("Lesson 000.23 diagnostic probe and octahedron exercise verified successfully!")